# Drive and utilities


In [1]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:
#@title Utils
!pip install dill
!pip install pandas
!pip install numpy
!
import dill as pickle

def load(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 2.5 MB/s eta 0:00:00


# Load data

In [3]:
#@title Load Data

!pip install numpy pandas scipy scikit-learn sklearn tqdm

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm
from scipy import stats

## Load
citations = pd.read_parquet("/content/drive/MyDrive/PhD Data/08 Citations/02 Patent citations - raw.parquet")
acquired_patents = pd.read_pickle("/content/drive/MyDrive/PhD Data/10 Sample - pre final/acquired_patents.pkl")  # if small, you can load as cuDF
potential_controls = pd.read_pickle("/content/drive/MyDrive/PhD Data/10 Sample - pre final/potential_controls.pkl")

## Remove GAFAM patents
clean_potential_ids = pd.read_csv("/content/drive/MyDrive/PhD Data/10 Sample - pre final/clean_potential_control_ids.csv")
clean_set = set(clean_potential_ids['patent_id'].astype(str))


## Remove
potential_controls_clean = potential_controls[potential_controls['patent_id'].isin(clean_set)]
assert len(potential_controls_clean) == len(clean_potential_ids)

# TRIM CITATIONS DATA
# Only keep citations for patents that are either treated or potential controls.
treated_ids = set(acquired_patents['patent_id'].unique())
control_ids = set(potential_controls_clean['patent_id'].unique())
valid_ids = treated_ids.union(control_ids)
citations = citations[citations['patent_id'].isin(valid_ids)]
print("Trimmed citations shape:", citations.shape)



  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Trimmed citations shape: (44962852, 3)


In [5]:
#@title Precompute Treated Patent Quarterly Counts
import pandas as pd
import numpy as np
import cudf
import pandas as pd
import numpy as np

# ================================================
# Step 1: Precompute Quarterly Citation Counts for All Patents (GPU version)
# ================================================
# Assume 'citations' is a cuDF DataFrame with columns:
#   - 'patent_id': the cited patent's ID
#   - 'citation_date': a datetime column

# Manually compute a quarter string: "YYYYQX"
citations['year'] = citations['citation_date'].dt.year
citations['month'] = citations['citation_date'].dt.month
# Compute quarter as integer: ((month-1)//3 + 1)
citations['qtr'] = ((citations['month'] - 1) // 3 + 1).astype(str)
citations['citation_quarter'] = citations['year'].astype(str) + "Q" + citations['qtr']

# Group by patent_id and citation_quarter to count citations per quarter.
# cuDF's groupby().size() returns a Series; unstack is not supported, so convert to pandas.
grouped = citations.groupby(['patent_id', 'citation_quarter']).size()
quarterly_counts_pd = grouped.unstack(fill_value=0)

# Build a dictionary mapping patent_id -> {quarter: count}
citation_counts_dict = {}
for pid, row in quarterly_counts_pd.iterrows():
    citation_counts_dict[pid] = row.to_dict()



In [6]:
#@title Rectangularize
import numpy as np
import pandas as pd
from datetime import datetime
import dateutil.relativedelta as rdelta
from scipy.linalg import block_diag

# ------------- Part A: Combine Data into a Rectangular Dataset ---------------

# --- Assume you have already loaded your datasets ---
# For example, you might have:
# acquired_patents = pd.read_pickle("acquired_patents.pkl")
# potential_controls = pd.read_pickle("potential_controls.pkl")
# citations = pd.read_parquet("citations.parquet")  # with columns: patent_id, citation_date, etc.

# For this example, we assume both treated and controls are in separate dataframes.
# We add a treatment indicator: treated = 1; potential_controls = 0.
acquired_patents['treat'] = 1
potential_controls['treat'] = 0

# Combine the two datasets into one.
df_all = pd.concat([acquired_patents, potential_controls], ignore_index=True)

# Convert date columns to datetime
df_all['acq_date'] = pd.to_datetime(df_all['acq_date'])
df_all['grant_date'] = pd.to_datetime(df_all['grant_date'])

In [11]:
df_all['acq_date'].isna().sum()

np.int64(2876736)

In [ ]:
#@title Functions

# --- Define functions to get pre- and post-treatment quarters ---
def get_quarter_str(date_obj):
    """Return a string like 'YYYYQX' for a given date."""
    year = date_obj.year
    # Quarter: 1 if month 1-3, 2 if 4-6, etc.
    quarter = (date_obj.month - 1) // 3 + 1
    return f"{year}Q{quarter}"

def add_quarters(date_obj, n):
    """Add n quarters to date_obj and return the new date."""
    # Each quarter is roughly 3 months
    return date_obj + rdelta.relativedelta(months=3*n)

def compute_pre_post_citations(row, citation_dict):
    """
    For a given patent row, compute:
      - pre_q4, pre_q3, pre_q2, pre_q1: citation counts for the 4 quarters immediately before acq_date.
      - post_q1, post_q2, ..., post_q16: citation counts for the 16 quarters after acq_date.

    citation_dict: a dictionary mapping patent_id (as a string) to a dictionary
                   {quarter_str: count}. If a count is missing, use zero.
    """
    acq_date = row['acq_date']

    # Compute pre-treatment citation counts
    pre_counts = {}
    for i in range(4, 0, -1):  # i = 4,3,2,1
        pre_date = add_quarters(acq_date, -i)
        q_str = get_quarter_str(pre_date)
        pre_counts[f"pre_q{i}"] = citation_dict.get(str(row['patent_id']), {}).get(q_str, 0)

    # Compute post-treatment citation counts for quarters 1 through 16 after acq_date
    post_counts = {}
    for i in range(1, 17):  # i = 1,2,...,16
        post_date = add_quarters(acq_date, i)
        q_str = get_quarter_str(post_date)
        post_counts[f"post_q{i}"] = citation_dict.get(str(row['patent_id']), {}).get(q_str, 0)

    # Combine pre- and post-treatment counts into one series.
    combined = {**pre_counts, **post_counts}
    return pd.Series(combined)


In [ ]:

# --- Apply the function to compute pre and post citations ---
pre_post_citations = df_all.apply(lambda row: compute_pre_post_citations(row, citation_counts_dict), axis=1)
df_all = pd.concat([df_all.reset_index(drop=True), pre_post_citations.reset_index(drop=True)], axis=1)

# --- Process Embeddings: Flatten the embedding list into separate columns ---
# Assume each embedding is a list or array of fixed dimension d.
def flatten_embedding(emb):
    # emb should be an iterable of numbers
    return pd.Series(emb)